# 1.5 dalis (a) — Perkeltasis mokymas (angl. *transfer learning*): požymių išskyrimas (angl. *feature extraction*)
### `„Transfer learning 1“ Colab stiliuje — MobileNetV2 + užšaldyta bazė`

**Kaip veikia originalus „Colab“ (Transfer learning 1):**
1. Imamas **iš anksto apmokytas** („ImageNet“) vaizdų modelis — čia `MobileNetV2`;
2. Bazinio modelio svoriai **užšaldomi** (`trainable = False`) — jo konvoliucijos
   jau „moka“ atpažinti kraštines, tekstūras ir formas;
3. Ant viršaus pridedama nauja klasifikavimo galvutė savoms klasėms;
4. Mokoma **tik galvutė** — naujos klasės išmokstamos iš kelių šimtų
   pavyzdžių, nes požymius teikia užšaldytas modelis.

**Mūsų atitikmuo:** tos pačios ketinimų klasės kaip ir 1.3–1.4 dalyse, bet vaizdai —
RGB (3 kanalai, 96×96) iš užklausų tekstų. Palyginimas: CNN, mokomas nuo
nulio (1.4 dalis), prieš modelį su perkeltaisiais „ImageNet“ svoriais.

> Rekomenduojama naudoti GPU: *Runtime → Change runtime type →* **T4 GPU**.


## 1. Aplinka ir duomenys

In [ ]:
%pip install -q tensorflow Pillow scikit-learn pandas matplotlib

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

SEED = 42
np.random.seed(SEED)

import tensorflow as tf
from PIL import Image, ImageDraw, ImageFont
import textwrap
tf.random.set_seed(SEED)
print('TensorFlow version:', tf.__version__)

In [ ]:
# --- Deterministic dataset generator (same as scripts/generate_dataset.py) ---
# X = customer query text, y = intent label (6 classes).
DATA_SPEC = {
    'order_status': [
        ('Kur mano užsakymas {order}?', 'order'),
        ('Kokia yra užsakymo {order} būsena?', 'order'),
        ('Nematau užsakymo {order} paskyroje, ką daryti?', 'order'),
        ('Kaip sužinoti mano užsakymo {order} statusą?', 'order'),
        ('Ar užsakymas {order} jau išsiųstas?', 'order'),
        ('Mano užsakymo {order} statusas vis dar „Apmokėta“, ar tai normalu?', 'order'),
        ('Kada bus matoma užsakymo {order} sekimo informacija?', 'order'),
    ],
    'delivery': [
        ('Kiek trunka pristatymas į {city}?', 'city'),
        ('Ar pristatote siuntas į {city} savaitgaliais?', 'city'),
        ('Kokia pristatymo į {city} kaina?', 'city'),
        ('Kada mano siunta pasieks {city}?', 'city'),
        ('Ar taikomas nemokamas pristatymas perkant už {eur} eurų?', 'eur'),
        ('Kur rasti siuntos į {city} sekimo numerį?', 'city'),
        ('Ar galiu atsiimti siuntą į {city} paštomate?', 'city'),
    ],
    'returns': [
        ('Noriu grąžinti {item}, kaip tai padaryti?', 'item'),
        ('Ar galiu grąžinti prekę po {days} dienų?', 'days'),
        ('Kiek kainuoja grąžinti {item}?', 'item'),
        ('Kada atgausiu pinigus grąžinęs {item}?', 'item'),
        ('Ar galima grąžinti {item} originalioje pakuotėje?', 'item'),
        ('Kur rasiu grąžinimo formą norėdamas grąžinti {item}?', 'item'),
        ('Ar galima grąžinti {item}, jei pakuotė jau buvo atidaryta?', 'item'),
    ],
    'payment': [
        ('Ar priimate {method}?', 'method'),
        ('Kiek laiko trunka mokėjimo autorizacija pasirinkus {method}?', 'method'),
        ('Nepavyko apmokėti užsakymo pasirinkus {method}, ką daryti?', 'method'),
        ('Ar galima mokėti dalimis pasirinkus {method}?', 'method'),
        ('Kada gausiu sąskaitą faktūrą apmokėjus per {method}?', 'method'),
        ('Ar palaikote {method} internetinėje parduotuvėje?', 'method'),
        ('Ką daryti, jei mokant per {method} nurašė pinigus, bet užsakymas neapmokėtas?', 'method'),
    ],
    'warranty': [
        ('Kiek laiko galioja garantija įsigijus {item}?', 'item'),
        ('Ar nemokamai suremontuosite {item} pagal garantiją?', 'item'),
        ('Ką daryti, jei gaminys sugedo po {days} dienų naudojimo?', 'days'),
        ('Ar reikia atskiro talono pateikiant {item} garantiniam remontui?', 'item'),
        ('Kiek trunka garantinis remontas pateikus {item}?', 'item'),
        ('Ar garantija galioja, jei netyčia pažeidžiau {item}?', 'item'),
        ('Ar atvežate {item} į remontą nemokamai?', 'item'),
    ],
    'product_usage': [
        ('Kaip susieti {item} su telefonu?', 'item'),
        ('Kur rasti naudotojo vadovą įsigijus {item}?', 'item'),
        ('Nusipirkau {item}, bet prietaisas neįsijungia, ką daryti?', 'item'),
        ('Kaip atstatyti gamyklinius nustatymus naudojant {item}?', 'item'),
        ('Ar galima gauti naudotojo vadovą lietuvių kalba įsigijus {item}?', 'item'),
        ('Kaip tinkamai prijungti {item} prie kompiuterio?', 'item'),
        ('Kiek laiko reikia krauti {item}?', 'item'),
    ],
}

SLOT_VALUES = {
    'order': ['TS-102345', 'TS-201111', 'TS-314560', 'TS-456789', 'TS-511235', 'TS-623456'],
    'city': ['Vilnių', 'Kauną', 'Klaipėdą', 'Šiaulius', 'Panevėžį', 'Alytų'],
    'item': ['išmanųjį telefoną', 'belaides ausines', 'išorinę bateriją',
             'elektroninę skaityklę', 'žaidimų pultelį', 'internetinę kamerą'],
    'days': ['3', '7', '10', '13', '20', '30'],
    'eur': ['30', '45', '60', '75', '90', '120'],
    'method': ['PayPal', 'Visa kortelę', 'Mastercard kortelę',
               'bankinį pavedimą', 'Google Pay', 'Apple Pay'],
}

INTENTS = ['order_status', 'delivery', 'returns', 'payment', 'warranty', 'product_usage']


def generate_dataset():
    rows = []
    for intent, specs in DATA_SPEC.items():
        for template, slot in specs:
            for value in SLOT_VALUES[slot]:
                rows.append((template.replace('{' + slot + '}', value), intent))
    return pd.DataFrame(rows, columns=['query', 'intent'])


df = generate_dataset()
print(f'Dataset: {len(df)} (X, y) pairs, {len(INTENTS)} classes')
print(df['intent'].value_counts())

# Canonical hold-out test set (same 6 queries as data/customer_support_test.csv
# and as the 1.2.2 Gemini prototype) - used to compare all models fairly.
TEST_EXAMPLES = [
    ('Kokiomis dienomis pristatote užsakymus į Vilnių?', 'delivery'),
    ('Norėčiau grąžinti prekę po 10 dienų nuo pirkimo, ar įmanoma?', 'returns'),
    ('Nematau savo užsakymo istorijos paskyroje, ką daryti?', 'order_status'),
    ('Ar parduotuvė priima PayPal mokėjimus?', 'payment'),
    ('Ar už 25 eurus pirktas gaminys turi garantiją?', 'warranty'),
    ('Kur rasti ausinių susiejimo instrukciją?', 'product_usage'),
]

IMG_SIZE = 96  # MobileNetV2 accepts >= 32x32 inputs; 96 keeps training fast

def get_font(size=16):
    for path in ['/usr/share/fonts/truetype/dejavu/DejaVuSans.ttf',
                 'C:/Windows/Fonts/arial.ttf']:
        try:
            return ImageFont.truetype(path, size)
        except Exception:
            continue
    return ImageFont.load_default()

def render_query_rgb(text, idx, img_size=IMG_SIZE):
    """Render query text as an RGB image (light background, dark text)."""
    rng = np.random.RandomState(idx)
    img = Image.new('RGB', (img_size, img_size), color=(235, 235, 235))
    draw = ImageDraw.Draw(img)
    wrapped = textwrap.wrap(text, width=13)[:4]
    y = 8 + rng.randint(-2, 3)
    for line in wrapped:
        draw.text((6 + rng.randint(-2, 3), y), line, fill=(20, 20, 20), font=get_font(15))
        y += 18
    return np.asarray(img, dtype='float32')

X_images = np.stack([render_query_rgb(q, i) for i, q in enumerate(df['query'])])
y_labels = np.array([INTENTS.index(l) for l in df['intent']])
print('RGB image dataset:', X_images.shape)
print()
plt.imshow(X_images[0].astype('uint8')); plt.axis('off')
plt.title(df['query'].iloc[0]); plt.show()

## 2. Užšaldytas „MobileNetV2“ + nauja galvutė

1. `MobileNetV2(include_top=False, weights='imagenet')` — be galutinio „ImageNet“
   klasifikatoriaus, su „ImageNet“ svoriais;
2. `Rescaling` sluoksnis pikselius perkelia į `[-1, 1]` intervalą, kokį modelis
   matė pirminio mokymosi metu;
3. `base_model.trainable = False` — svoriai užšaldyti, mokysis tik galvutė:

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_val, y_train, y_val_raw = train_test_split(
    X_images, df['intent'], test_size=0.2, random_state=SEED, stratify=df['intent'])

y_train = np.array([INTENTS.index(l) for l in y_train])
y_val = np.array([INTENTS.index(l) for l in y_val_raw])

X_test = np.stack([render_query_rgb(q, 20_000 + i) for i, (q, _) in enumerate(TEST_EXAMPLES)])
y_test = np.array([INTENTS.index(l) for _, l in TEST_EXAMPLES])

base_model = tf.keras.applications.MobileNetV2(
    input_shape=(IMG_SIZE, IMG_SIZE, 3), include_top=False, weights='imagenet')
base_model.trainable = False  # freeze pretrained weights

inputs = tf.keras.Input(shape=(IMG_SIZE, IMG_SIZE, 3))
x = tf.keras.layers.Rescaling(1.0 / 127.5, offset=-1.0)(inputs)  # MobileNetV2 expects [-1, 1]
x = base_model(x, training=False)
x = tf.keras.layers.GlobalAveragePooling2D()(x)
x = tf.keras.layers.Dropout(0.3)(x)
outputs = tf.keras.layers.Dense(len(INTENTS), activation='softmax')(x)
tl_model = tf.keras.Model(inputs, outputs)

tl_model.compile(optimizer=tf.keras.optimizers.Adam(1e-3),
                 loss='sparse_categorical_crossentropy',
                 metrics=['accuracy'])
tl_model.summary()
print(f'Trainable variables: {len(tl_model.trainable_variables)} (tik galvutė)')
print(f'Non-trainable variables: {len(tl_model.non_trainable_variables)} (užšaldyta bazė)')

## 3. Mokymasis (tik galvutė)

In [ ]:
early_stop = tf.keras.callbacks.EarlyStopping(
    monitor='val_accuracy', patience=5, restore_best_weights=True)

tl_history = tl_model.fit(
    X_train, y_train,
    validation_data=(X_val, y_val),
    epochs=15, batch_size=32,
    callbacks=[early_stop], verbose=2,
)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(tl_history.history['loss'], label='mokymo nuostoliai')
axes[0].plot(tl_history.history['val_loss'], label='validacijos nuostoliai')
axes[0].set_xlabel('Epocha'); axes[0].set_ylabel('Nuostoliai (Loss)'); axes[0].legend(); axes[0].grid(alpha=0.3)
axes[1].plot(tl_history.history['accuracy'], label='mokymo tikslumas')
axes[1].plot(tl_history.history['val_accuracy'], label='validacijos tikslumas')
axes[1].set_xlabel('Epocha'); axes[1].set_ylabel('Tikslumas (Accuracy)'); axes[1].legend(); axes[1].grid(alpha=0.3)
plt.suptitle('Perkeltojo mokymo (požymių išskyrimo) mokymosi kreivės')
plt.tight_layout(); plt.show()

## 4. Vertinimas

In [ ]:
def plot_confusion_matrix(y_true, y_pred, classes, title):
    cm = np.zeros((len(classes), len(classes)), dtype=int)
    for t, p in zip(y_true, y_pred):
        cm[t, p] += 1
    fig, ax = plt.subplots(figsize=(5.5, 5))
    im = ax.imshow(cm, cmap='Blues')
    ax.set_xticks(range(len(classes)), classes, rotation=45, ha='right')
    ax.set_yticks(range(len(classes)), classes)
    for i in range(len(classes)):
        for j in range(len(classes)):
            ax.text(j, i, cm[i, j], ha='center', va='center',
                    color='white' if cm[i, j] > cm.max() / 2 else 'black')
    ax.set_xlabel('Numatytas ketinimas')
    ax.set_ylabel('Tikrasis ketinimas')
    ax.set_title(title)
    fig.colorbar(im)
    plt.tight_layout()
    plt.show()
    return cm

from sklearn.metrics import accuracy_score

val_pred = np.argmax(tl_model.predict(X_val, verbose=0), axis=1)
test_pred = np.argmax(tl_model.predict(X_test, verbose=0), axis=1)

print(f'Validacijos tikslumas: {accuracy_score(y_val, val_pred):.4f}')
print(f'Testavimo (6 užklausos) tikslumas: {accuracy_score(y_test, test_pred):.4f}')
print()
for (query, true_label), pred_id in zip(TEST_EXAMPLES, test_pred):
    mark = 'OK ' if INTENTS[pred_id] == true_label else 'WRONG'
    print(f'{mark} true={true_label:14s} pred={INTENTS[pred_id]:14s} {query}')

plot_confusion_matrix(y_val, val_pred, INTENTS,
                      'Perkeltasis mokymas (požymių išskyrimas) — painiavos matrica')

## 5. Išvados
1. Užšaldyta „ImageNet“ bazė suteikia universalias vaizdo požymių formas, todėl
   net ir mokant tik mažą klasifikavimo galvutę modelis greitai pasiekia gerus rezultatus
   su nedideliu savų duomenų kiekiu — tai pagrindinė perkeltojo mokymo (angl. *transfer learning*) idėja.
2. Palyginus su 1.4 dalies CNN nuo nulio: mokymasis dažnai būna stabilesnis
   (mažesnis nuostolių svyravimas), nes didžioji dalis svorių nekeičiama.
3. Kitas žingsnis — 1.5(b): bazinio modelio viršutinių sluoksnių
   **atšaldymas ir tikslusis derinimas** (angl. *fine-tuning*) su mažesniu mokymosi greičiu.
